[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_14/SFM_ch14_stablecoins/SFM_ch14_stablecoins.ipynb)

# SFM_ch14_stablecoins

Stablecoins: supply of USD stablecoins, USDT, USDC and DAI (DefiLlama); peg deviations in basis points, d = 10 000 (P - 1), on daily closes and lows since 2021, with the AR(1) half-life of a deviation; the USDC and DAI depeg of March 2023 after the closure of Silicon Valley Bank; the collapse of TerraUSD (UST) in May 2022; weekly changes in stablecoin supply against Bitcoin returns (HAC regression).

*Statistics of Financial Markets (SFM), Chapter 14: Crypto assets. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_14'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
import matplotlib.dates as mdates

import statsmodels.api as sm

from arch import arch_model

EXTRA = {'usdc': ('USDC-USD.CC', 'USD Coin (USDC)', 'Crypto', 'close', '2018-10-09'), 'dai': ('DAI-USD.CC', 'Dai (DAI)', 'Crypto', 'close', '2017-12-27'), 'ibit': ('IBIT.US', 'iShares Bitcoin Trust (IBIT)', 'ETF', 'adjusted_close', '2024-01-11')}

NAME = {'btc': 'Bitcoin', 'eth': 'Ethereum', 'sol': 'Solana', 'sp500': 'S&P 500', 'gold': 'Gold', 'usdt': 'Tether (USDT)', 'usdc': 'USD Coin (USDC)', 'dai': 'Dai (DAI)', 'ibit': 'IBIT'}

START = {'btc': '2014-09-17', 'eth': '2016-01-01', 'sol': '2020-04-11', 'sp500': '2014-09-17', 'gold': '2014-09-17'}

ASSETS = ['btc', 'eth', 'sol', 'sp500', 'gold']

COLORS = {'btc': '#E67E22', 'eth': '#8E44AD', 'sol': '#17A2B8', 'sp500': '#1A3A6E', 'gold': '#B5853F', 'usdt': '#2E7D32', 'usdc': '#1A3A6E', 'dai': '#E67E22', 'total': '#CD0000', 'ibit': '#1A3A6E'}

HALVINGS = ['2012-11-28', '2016-07-09', '2020-05-11', '2024-04-20']

ETF_DATE = '2024-01-11'

EVENTS = {'COVID-19': '2020-03-12', 'Terra/Luna': '2022-05-09', 'FTX': '2022-11-08', 'spot ETFs': '2024-01-11'}

EPISODES = {'COVID-19': ('2020-02-19', '2020-03-23'), 'Terra/Luna': ('2022-05-05', '2022-05-12'), 'FTX': ('2022-11-05', '2022-11-21')}

HILL_FRAC = 0.025

VR_Q = (2, 5, 10)

VR_WINDOW, VR_STEP = 500, 21

SUBPERIODS = [('2014-09-17', '2017-12-31'), ('2018-01-01', '2021-12-31'), ('2022-01-01', '2026-09-18')]

CORR_WINDOW = 250

ALPHA_VAR, ALPHA_ES = 0.01, 0.025

POSITION = 100000

BT_START = '2018-01-01'

BT_WINDOW = 365

PEG_START = '2021-01-01'

SVB = ('2023-03-08', '2023-03-20')

LLAMA = 'https://stablecoins.llama.fi/stablecoincharts/all'

STABLE_ID = {'usdt': 1, 'usdc': 2, 'ust': 3, 'dai': 5}

DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

WD_PERIODS = {'2014-2019': ('2014-09-17', '2019-12-31'), '2020-2023': ('2020-01-01', '2023-12-31'), '2024-2026': ('2024-01-01', '2026-09-18')}

SEED = 2026

_LLAMA = {}

TABLE_DIR = '.'

def register_extra():
    """Add USDC, DAI and the IBIT ETF to the list of named series of the course data loader."""
    for k, v in EXTRA.items():
        SERIES.setdefault(k, v)


def close(k, start=None, end=END):
    """Daily close (adjusted close for the ETF) on the series' own calendar (crypto: 7 days a week)."""
    register_extra()
    return load_close(k, start or START.get(k), end)


def returns(k, start=None, end=END):
    """Daily log returns in % on the series' own calendar."""
    register_extra()
    return log_returns(k, start or START.get(k), end)


def joint_returns(a, b, start=None, end=END):
    """Log returns in % of two series on their common days: prices are joined first, then differenced."""
    p = pd.concat([close(a, start, end), close(b, start, end)], axis=1).dropna()
    return (100 * np.log(p).diff()).dropna()


def hac_ols(y, X, lags=10):
    """OLS with Newey-West (HAC) standard errors."""
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})


def stablecoin_chart(sid=None):
    """DefiLlama: circulating supply of USD stablecoins, billion USD (sid=None: all; 1 USDT, 2 USDC, 3 UST, 5 DAI);
    columns 'supply' (billion units) and 'value' (billion USD)."""
    if sid in _LLAMA:
        return _LLAMA[sid]
    url = LLAMA + (f'?stablecoin={sid}' if sid else '')
    js = json.loads(urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                           timeout=120).read())
    rows = {}
    for x in js:
        d = pd.Timestamp(int(x['date']), unit='s').normalize()
        c = (x.get('totalCirculating') or {}).get('peggedUSD')
        v = (x.get('totalCirculatingUSD') or {}).get('peggedUSD')
        if c is not None:
            rows[d] = (c / 1e9, (v if v is not None else np.nan) / 1e9)
    df = pd.DataFrame(rows, index=['supply', 'value']).T.sort_index().loc[:END]
    _LLAMA[sid] = df
    return df


def stablecoin_prices(coin='terrausd', start='2022-04-01', end='2022-06-30'):
    """DefiLlama: daily USD price of one stablecoin (key of the 'prices' dictionary, e.g. 'terrausd')."""
    key = ('prices', coin)
    if key not in _LLAMA:
        js = json.loads(urllib.request.urlopen(urllib.request.Request(
            'https://stablecoins.llama.fi/stablecoinprices', headers={'User-Agent': 'Mozilla'}), timeout=120).read())
        s = pd.Series({pd.Timestamp(int(x['date']), unit='s').normalize(): x['prices'][coin]
                       for x in js if coin in (x.get('prices') or {})}).sort_index()
        _LLAMA[key] = s
    return _LLAMA[key].loc[start:end]


def fig_stablecoin_supply(save=True):
    """Supply of USD stablecoins (DefiLlama): total, USDT, USDC, DAI, billion USD."""
    tot = stablecoin_chart()['value'].loc['2019-01-01':]
    parts = {k: stablecoin_chart(STABLE_ID[k])['value'].loc['2019-01-01':] for k in ('usdt', 'usdc', 'dai')}
    fig, ax = plt.subplots(figsize=(11, 3.9))
    ax.plot(tot.index, tot.values, color=COLORS['total'], lw=1.5, label='All USD stablecoins')
    for k, s in parts.items():
        ax.plot(s.index, s.values, color=COLORS[k], lw=1.2, label=NAME[k])
    ax.set_ylabel('Supply (USD bn)')
    st.legend_outside_bottom(ax, ncol=4, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_stablecoin_supply')
    lst = json.loads(urllib.request.urlopen(urllib.request.Request(
        'https://stablecoins.llama.fi/stablecoins?includePrices=false', headers={'User-Agent': 'Mozilla'}), timeout=120).read())
    mech = {}
    for x in lst['peggedAssets']:
        if x.get('pegType') == 'peggedUSD':
            m = (x.get('pegMechanism') or 'other').replace('crytpo', 'crypto')
            mech[m] = mech.get(m, 0) + ((x.get('circulating') or {}).get('peggedUSD') or 0) / 1e9
    end = tot.iloc[-1]
    y = lambda s, d: float(s.loc[:d].iloc[-1])   # noqa: E731
    return {'total_end': float(end), 'end': tot.index[-1].date().isoformat(), 'total_2020': y(tot, '2020-01-01'),
            'total_2022max': float(tot.loc[:'2022-12-31'].max()), 'total_2022max_date': tot.loc[:'2022-12-31'].idxmax().date().isoformat(),
            'total_trough': float(tot.loc['2022-06-01':'2024-06-30'].min()),
            'total_trough_date': tot.loc['2022-06-01':'2024-06-30'].idxmin().date().isoformat(),
            'usdt_end': float(parts['usdt'].iloc[-1]), 'usdc_end': float(parts['usdc'].iloc[-1]), 'dai_end': float(parts['dai'].iloc[-1]),
            'usdt_share': float(100 * parts['usdt'].iloc[-1] / end), 'usdc_share': float(100 * parts['usdc'].iloc[-1] / end),
            'usdc_2023_03_10': y(parts['usdc'], '2023-03-10'), 'usdc_2023_12_31': y(parts['usdc'], '2023-12-31'),
            'growth_x': float(end / y(tot, '2020-01-01')), 'mech_today': {k: float(v) for k, v in mech.items()}}


def peg_stats(k, start=PEG_START, end=END):
    """Peg deviation in basis points, d_t = 10 000 (P_t - 1), on daily closes and daily lows; AR(1) coefficient of
    the closing deviation and the half-life ln(0.5)/ln(phi) of a deviation."""
    register_extra()
    d = read_market(SERIES[k][0]).loc[start:end]
    dev = 1e4 * (d['close'] - 1)
    low = 1e4 * (d['low'] - 1)
    x = dev.values
    phi = np.corrcoef(x[1:], x[:-1])[0, 1]
    return {'n': int(len(dev)), 'mean': float(dev.mean()), 'sd': float(dev.std()), 'mad': float(np.median(np.abs(dev))),
            'gt10': float(100 * np.mean(np.abs(dev) > 10)), 'gt50': float(100 * np.mean(np.abs(dev) > 50)),
            'gt100': float(100 * np.mean(np.abs(dev) > 100)), 'close_min': float(dev.min()),
            'close_min_date': dev.idxmin().date().isoformat(), 'close_max': float(dev.max()),
            'close_max_date': dev.idxmax().date().isoformat(), 'low_min': float(low.min()),
            'low_min_date': low.idxmin().date().isoformat(), 'phi': float(phi),
            'half_life': float(np.log(0.5) / np.log(abs(phi)))}


def fig_peg(save=True):
    """Daily closing deviation from 1 USD (basis points) of USDT, USDC and DAI since 2021."""
    fig, axes = plt.subplots(3, 1, figsize=(11, 5.4), sharex=True)
    out = {}
    register_extra()
    for ax, k in zip(axes, ('usdt', 'usdc', 'dai')):
        d = read_market(SERIES[k][0]).loc[PEG_START:END]
        dev = 1e4 * (d['close'] - 1)
        ax.plot(dev.index, dev.clip(lower=-400).values, color=COLORS[k], lw=0.9, label=f'{NAME[k]}: close')
        ax.axhline(0, color='black', lw=0.5)
        ax.set_ylim(-310, 130)
        ax.set_ylabel('bp')
        out[k] = peg_stats(k)
    st.fig_legend_bottom(fig, ncol=3, y=0.0)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_peg')
    return out


def fig_svb(save=True):
    """March 2023: USDC and DAI below the peg after the closure of Silicon Valley Bank (10 March); daily closes and
    lows; USDT for comparison."""
    register_extra()
    a, b = SVB
    fig, ax = plt.subplots(figsize=(11, 3.9))
    out = {}
    for k in ('usdc', 'dai', 'usdt'):
        d = read_market(SERIES[k][0]).loc[a:b]
        ax.plot(d.index, d['close'], 'o-', color=COLORS[k], ms=4, lw=1.3, label=f'{NAME[k]}: close')
        ax.plot(d.index, d['low'], 'v', color=COLORS[k], ms=6, label=f'{NAME[k]}: daily low')
        out[k] = {'low': float(d['low'].min()), 'low_date': d['low'].idxmin().date().isoformat(),
                  'close_min': float(d['close'].min()), 'close_min_date': d['close'].idxmin().date().isoformat(),
                  'close_max': float(d['close'].max()), 'close_max_date': d['close'].idxmax().date().isoformat(),
                  'closes': {i.date().isoformat(): float(v) for i, v in d['close'].items()}}
    ax.axhline(1, color='black', lw=0.7, ls='--')
    ax.set_ylabel('Price (USD)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    st.legend_outside_bottom(ax, ncol=3, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_svb')
    return out


def fig_ust(save=True):
    """TerraUSD (UST), April-June 2022 (DefiLlama): circulating supply (until the last published day) and price."""
    u = stablecoin_chart(STABLE_ID['ust']).loc['2022-04-01':'2022-06-15']
    u = u[u['supply'] > 0]['supply']
    px = stablecoin_prices('terrausd', '2022-04-01', '2022-06-15')
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
    axes[0].plot(u.index, u.values, 'o-', color=st.MainBlue, ms=2.5, lw=1.4, label='UST supply (billion)')
    axes[0].set_ylabel('Billion UST')
    axes[1].plot(px.index, px.values, 'o-', color=st.IDAred, ms=2.5, lw=1.2, label='UST price (USD)')
    axes[1].axhline(1, color='black', lw=0.7, ls='--')
    axes[1].set_ylabel('USD')
    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
        ax.tick_params(axis='x', labelsize=10, rotation=30)
    st.fig_legend_bottom(fig, ncol=2, y=0.0)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_ust')
    pre = px.loc[:'2022-05-07']
    return {'supply_peak': float(u.max()), 'supply_peak_date': u.idxmax().date().isoformat(),
            'supply_last': float(u.iloc[-1]), 'last_date': u.index[-1].date().isoformat(),
            'supply_drop': float(100 * (1 - u.iloc[-1] / u.max())),
            'px_pre_min': float(pre.min()), 'px_pre_max': float(pre.max()),
            'px_0509': float(px.loc['2022-05-09']), 'px_0513': float(px.loc['2022-05-13']),
            'px_0514': float(px.loc['2022-05-14']), 'px_0531': float(px.loc['2022-05-31']),
            'px_end': float(px.iloc[-1]), 'px_end_date': px.index[-1].date().isoformat()}


def stablecoin_flows_btc(start='2020-01-01'):
    """Weekly (Sunday to Sunday) log change of the total USD stablecoin supply against Bitcoin's log return in the
    same week and in the next week; OLS with Newey-West standard errors (4 lags)."""
    s = stablecoin_chart()['value'].loc[start:END].resample('W-SUN').last()
    p = close('btc', start).resample('W-SUN').last()
    d = pd.concat([100 * np.log(s).diff(), 100 * np.log(p).diff()], axis=1, keys=['ds', 'r']).dropna()
    d['r_next'] = d['r'].shift(-1)
    d = d.dropna()
    reg = hac_ols(d['r_next'], d['ds'], lags=4)
    return {'n': int(len(d)), 'corr_same': float(d['ds'].corr(d['r'])), 'corr_next': float(d['ds'].corr(d['r_next'])),
            'b': float(reg.params['ds']), 't': float(reg.tvalues['ds']), 'p': float(reg.pvalues['ds']), 'r2': float(reg.rsquared)}

## Run

In [ ]:
print(fig_stablecoin_supply())
print(pd.DataFrame(fig_peg()).T.round(3))
print(fig_svb())
print(fig_ust())
print(stablecoin_flows_btc())

![sfm_ch14_stablecoin_supply](./sfm_ch14_stablecoin_supply.png)

Output: `sfm_ch14_stablecoin_supply.pdf`

![sfm_ch14_peg](./sfm_ch14_peg.png)

Output: `sfm_ch14_peg.pdf`

![sfm_ch14_svb](./sfm_ch14_svb.png)

Output: `sfm_ch14_svb.pdf`

![sfm_ch14_ust](./sfm_ch14_ust.png)

Output: `sfm_ch14_ust.pdf`